# FootyStats — Perfil de Gols apos Minuto 60 por Equipe

Para cada time das 29 ligas, analisa o comportamento de gols apos o minuto 60.

**Regras:**
- Base = TODOS os jogos finalizados (incluindo 0x0)
- Filtramos todos os gols do time com `minuto > 60`
- Jogos sem gol apos min 60 entram no denominador normalmente

**Colunas geradas:**
- `pct_marcou_M60_casa/fora` — % de jogos com pelo menos 1 gol apos min 60
- `avg/dp/cv_min_M60_casa/fora` — estatisticas do 1o gol apos min 60 (so jogos onde ocorreu)
- `%Gl_Ca/Fo_61_70M`, `71_80M`, `81_90M` — % de jogos com gol em cada faixa (base = total jogos)

## 1. Imports

In [1]:
import time
import requests
import pandas as pd
import numpy as np

## 2. Configuracao

In [2]:
API_KEY      = "033f6d5fc7b707b144fbd2c28898c1540eb066ee9efea6c038d740ffedeacb5f"
URL_MATCHES  = "https://api.football-data-api.com/league-matches"
MAX_PER_PAGE = 200
DELAY        = 1.0
MIN_CORTE    = 60   # gols apos esse minuto

## 3. Lista de Ligas

In [3]:
LIGAS = [
    {"liga": "Alemanha_1 Bundesliga",        "season_id": 14968},
    {"liga": "Alemanha_2 2. Bundesliga",     "season_id": 14931},
    {"liga": "Alemanha_3 3. Liga",           "season_id": 14977},
    {"liga": "Australia_1 A-League",         "season_id": 16036},
    {"liga": "Belgica_1 Pro League",         "season_id": 14937},
    {"liga": "Bulgaria_1 First League",      "season_id": 15056},
    {"liga": "Croacia_1 HNL",               "season_id": 15053},
    {"liga": "Dinamarca_1 Superliga",        "season_id": 15055},
    {"liga": "Escocia_1 Premiership",        "season_id": 15000},
    {"liga": "Espanha_1 La Liga",            "season_id": 14956},
    {"liga": "Espanha_2 Segunda Division",   "season_id": 15066},
    {"liga": "Franca_1 Ligue 1",            "season_id": 14932},
    {"liga": "Franca_2 Ligue 2",            "season_id": 14954},
    {"liga": "Grecia_1 Super League",        "season_id": 15163},
    {"liga": "Holanda_1 Eredivisie",         "season_id": 14936},
    {"liga": "Holanda_2 Eerste Divisie",     "season_id": 14987},
    {"liga": "Hungria_1 NB I",              "season_id": 14963},
    {"liga": "Inglaterra_1 Premier League",  "season_id": 15050},
    {"liga": "Inglaterra_2 Championship",    "season_id": 14930},
    {"liga": "Inglaterra_3 League One",      "season_id": 14934},
    {"liga": "Italia_1 Serie A",             "season_id": 15068},
    {"liga": "Italia_2 Serie B",             "season_id": 15632},
    {"liga": "Portugal_1 Primeira Liga",     "season_id": 15115},
    {"liga": "Rep_Tcheca_1 Czech Liga",      "season_id": 14973},
    {"liga": "Turquia_1 Super Lig",          "season_id": 14972},
    {"liga": "Argentina_1 Liga Profesional", "season_id": 16571},
    {"liga": "Brasileiro_A Serie A",         "season_id": 16544},
    {"liga": "Chile_1 Primera Division",     "season_id": 16615},
    {"liga": "Colombia_1 Liga BetPlay",      "season_id": 16614},
]
print(f"Total de ligas: {len(LIGAS)}")

Total de ligas: 29


## 4. Funcao de Busca

In [4]:
def buscar_todos_jogos(season_id: int, api_key: str) -> list:
    """
    Busca TODOS os jogos finalizados da liga (todas as paginas).
    Inclui jogos 0x0 — necessario para denominador correto.
    Nao filtra por goal_timings_recorded aqui pois 0x0 tem timing vazio mas e valido.
    """
    todos = []
    page  = 1
    while True:
        resp = requests.get(
            URL_MATCHES,
            params={"key": api_key, "season_id": season_id,
                    "max_per_page": MAX_PER_PAGE, "page": page},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        if not data.get("success"):
            break
        jogos    = data.get("data", [])
        pager    = data.get("pager", {})
        max_page = pager.get("max_page", 1)
        # Todos finalizados, inclusive 0x0
        todos.extend([j for j in jogos if j.get("status") == "complete"])
        if page >= max_page:
            break
        page += 1
        time.sleep(DELAY)
    return todos

## 5. Funcoes de Calculo

In [5]:
# Faixas apos min 60
FAIXAS_M60 = [
    ("61_70", 61,  70),
    ("71_80", 71,  80),
    ("81_90", 81, 999),
]


def parse_timings(raw) -> list:
    """Converte lista de minutos para inteiros. Trata acrescimos: '90+3' -> 93."""
    if not raw:
        return []
    result = []
    for v in raw:
        try:
            s = str(v)
            if '+' in s:
                parts = s.split('+')
                result.append(int(parts[0]) + int(parts[1]))
            else:
                result.append(int(s))
        except:
            pass
    return result


def calcular_perfil_M60(team_name: str, jogos: list, min_corte: int = MIN_CORTE) -> dict:
    """
    Para um time, percorre TODOS os jogos finalizados e calcula:
    - % de jogos com pelo menos 1 gol marcado apos min_corte (base = total jogos)
    - Estatisticas (avg, dp, cv) do 1o gol apos min_corte (so nos jogos onde ocorreu)
    - % de jogos com gol em cada faixa 61-70, 71-80, 81-90 (base = total jogos)
    """
    # Listas de minutos do 1o gol apos corte (por condicao)
    primeiro_M60_casa = []  # minuto do 1o gol >60 nos jogos em casa
    primeiro_M60_fora = []  # minuto do 1o gol >60 nos jogos fora

    # Todos os gols >60 (para calculo de faixa)
    todos_M60_casa = []
    todos_M60_fora = []

    total_casa = 0
    total_fora = 0

    for j in jogos:
        e_casa = j.get("home_name") == team_name
        e_fora = j.get("away_name") == team_name
        if not e_casa and not e_fora:
            continue

        if e_casa:
            total_casa += 1
            meus = parse_timings(j.get("homeGoals_timings") or j.get("homeGoals"))
        else:
            total_fora += 1
            meus = parse_timings(j.get("awayGoals_timings") or j.get("awayGoals"))

        # Filtrar gols apos o corte
        gols_pos = sorted([m for m in meus if m > min_corte])

        if e_casa:
            todos_M60_casa.extend(gols_pos)
            if gols_pos:
                primeiro_M60_casa.append(gols_pos[0])  # 1o gol apos corte
        else:
            todos_M60_fora.extend(gols_pos)
            if gols_pos:
                primeiro_M60_fora.append(gols_pos[0])

    # ── Funcoes estatisticas ─────────────────────────────────────────
    def media(lst):
        return round(float(np.mean(lst)), 1) if lst else np.nan

    def dp(lst):
        return round(float(np.std(lst, ddof=1)), 1) if len(lst) >= 2 else np.nan

    def cv(lst):
        if len(lst) < 2:
            return np.nan
        m = np.mean(lst)
        return round(float(np.std(lst, ddof=1) / m * 100), 1) if m != 0 else np.nan

    def pct(n, total):
        """% sobre total de jogos (denominador honesto)."""
        return round(n / total * 100, 1) if total > 0 else np.nan

    def pct_faixa(lst_todos_gols, total_jogos, low, high):
        """
        % de jogos com pelo menos 1 gol na faixa [low, high].
        Base = total de jogos disputados (incluindo 0x0 e sem gol pos-60).
        """
        if total_jogos == 0:
            return np.nan
        # Conta jogos distintos — aproximacao: conta gols na faixa como eventos independentes
        # Para exatidao, contariamos por jogo, mas os timings aqui sao agregados
        n = sum(1 for m in lst_todos_gols if low <= m <= high)
        return round(n / total_jogos * 100, 1)

    resultado = {
        # Total de jogos (denominador)
        "total_jogos_casa"         : total_casa,
        "total_jogos_fora"         : total_fora,
        # % de jogos com pelo menos 1 gol apos min 60
        "pct_marcou_M60_casa"      : pct(len(primeiro_M60_casa), total_casa),
        "pct_marcou_M60_fora"      : pct(len(primeiro_M60_fora), total_fora),
        # Estatisticas do 1o gol apos min 60 (so jogos onde ocorreu)
        "avg_min_M60_casa"         : media(primeiro_M60_casa),
        "avg_min_M60_fora"         : media(primeiro_M60_fora),
        "dp_min_M60_casa"          : dp(primeiro_M60_casa),
        "dp_min_M60_fora"          : dp(primeiro_M60_fora),
        "cv_min_M60_casa"          : cv(primeiro_M60_casa),
        "cv_min_M60_fora"          : cv(primeiro_M60_fora),
        # n bruto para validacao
        "n_marcou_M60_casa"        : len(primeiro_M60_casa),
        "n_marcou_M60_fora"        : len(primeiro_M60_fora),
    }

    # % por faixa de 10 em 10 apos min 60 (base = total jogos)
    for label, low, high in FAIXAS_M60:
        resultado[f"%Gl_Ca_{label}M"] = pct_faixa(todos_M60_casa, total_casa, low, high)
        resultado[f"%Gl_Fo_{label}M"] = pct_faixa(todos_M60_fora, total_fora, low, high)

    return resultado

## 6. Funcao Principal

In [6]:
def buscar_perfil_M60_todas_ligas(
    api_key : str  = API_KEY,
    ligas   : list = LIGAS,
) -> pd.DataFrame:
    """
    Para cada liga, busca todos os jogos e calcula o perfil
    de gols apos min 60 por time (casa e fora).
    Retorna DataFrame com uma linha por time.
    """
    frames = []
    erros  = []

    for i, entry in enumerate(ligas, 1):
        liga_nome = entry["liga"]
        season_id = entry["season_id"]

        print(f"[{i:02d}/{len(ligas)}] {liga_nome} (season_id={season_id})")

        try:
            jogos = buscar_todos_jogos(season_id, api_key)

            # Extrair nomes unicos de times
            times = sorted(set(
                [j["home_name"] for j in jogos if j.get("home_name")] +
                [j["away_name"] for j in jogos if j.get("away_name")]
            ))

            print(f"  -> {len(jogos)} jogos | {len(times)} times")

            linhas = []
            for nome_time in times:
                perfil = calcular_perfil_M60(nome_time, jogos)

                linha = {
                    "Liga"  : liga_nome,
                    "time"  : nome_time,
                    # Total de jogos
                    "total_jogos_casa"    : perfil["total_jogos_casa"],
                    "total_jogos_fora"    : perfil["total_jogos_fora"],
                    # % com gol apos min 60
                    "pct_marcou_M60_casa" : perfil["pct_marcou_M60_casa"],
                    "pct_marcou_M60_fora" : perfil["pct_marcou_M60_fora"],
                    # Media do 1o gol apos min 60
                    "avg_min_M60_casa"    : perfil["avg_min_M60_casa"],
                    "avg_min_M60_fora"    : perfil["avg_min_M60_fora"],
                    # Desvio Padrao
                    "dp_min_M60_casa"     : perfil["dp_min_M60_casa"],
                    "dp_min_M60_fora"     : perfil["dp_min_M60_fora"],
                    # Coeficiente de Variacao %
                    "cv_min_M60_casa"     : perfil["cv_min_M60_casa"],
                    "cv_min_M60_fora"     : perfil["cv_min_M60_fora"],
                    # Contagens brutas
                    "n_marcou_M60_casa"   : perfil["n_marcou_M60_casa"],
                    "n_marcou_M60_fora"   : perfil["n_marcou_M60_fora"],
                }

                # Colunas dinamicas %Gl_*
                for k, v in perfil.items():
                    if k.startswith("%Gl_"):
                        linha[k] = v

                linhas.append(linha)

            df_liga = pd.DataFrame(linhas)
            frames.append(df_liga)
            print(f"  OK: {len(linhas)} times")

        except requests.exceptions.HTTPError as e:
            print(f"  ERRO HTTP {e.response.status_code}: {e}")
            erros.append({"liga": liga_nome, "erro": str(e)})

        except Exception as e:
            print(f"  ERRO: {e}")
            erros.append({"liga": liga_nome, "erro": str(e)})

        time.sleep(DELAY)

    print(f"\n{'-'*55}")
    print(f"Ligas OK      : {len(frames)}/{len(ligas)}")
    print(f"Ligas c/ erro : {len(erros)}")
    if erros:
        for e in erros:
            print(f"  - {e['liga']}: {e['erro']}")

    if not frames:
        return pd.DataFrame()

    df_final = (
        pd.concat(frames, ignore_index=True)
        .sort_values(["Liga", "time"])
        .reset_index(drop=True)
    )
    print(f"\nDataFrame final: {df_final.shape[0]} times x {df_final.shape[1]} colunas")
    print(f"Colunas: {df_final.columns.tolist()}")
    return df_final

## 7. Execucao

In [7]:
df_M60 = buscar_perfil_M60_todas_ligas()

[01/29] Alemanha_1 Bundesliga (season_id=14968)
  -> 243 jogos | 18 times
  OK: 18 times
[02/29] Alemanha_2 2. Bundesliga (season_id=14931)
  -> 243 jogos | 18 times
  OK: 18 times
[03/29] Alemanha_3 3. Liga (season_id=14977)
  -> 300 jogos | 20 times
  OK: 20 times
[04/29] Australia_1 A-League (season_id=16036)
  -> 131 jogos | 12 times
  OK: 12 times
[05/29] Belgica_1 Pro League (season_id=14937)
  -> 240 jogos | 16 times
  OK: 16 times
[06/29] Bulgaria_1 First League (season_id=15056)
  -> 216 jogos | 16 times
  OK: 16 times
[07/29] Croacia_1 HNL (season_id=15053)
  -> 135 jogos | 10 times
  OK: 10 times
[08/29] Dinamarca_1 Superliga (season_id=15055)
  -> 144 jogos | 12 times
  OK: 12 times
[09/29] Escocia_1 Premiership (season_id=15000)
  -> 186 jogos | 12 times
  OK: 12 times
[10/29] Espanha_1 La Liga (season_id=14956)
  -> 290 jogos | 20 times
  OK: 20 times
[11/29] Espanha_2 Segunda Division (season_id=15066)
  -> 341 jogos | 22 times
  OK: 22 times
[12/29] Franca_1 Ligue 1 (se

## 8. Visualizacao

In [8]:
import warnings
warnings.filterwarnings('ignore')
pd.options.mode.copy_on_write = True 
pd.set_option('display.max_columns', None)

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

In [9]:
df_M60.to_csv("0_PAINEL_EVOLUTION_JOGOS_GOLS_M60_API.csv", index=False)

In [10]:
df_M60.head(5)

,Liga,time,total_jogos_casa,total_jogos_fora,pct_marcou_M60_casa,pct_marcou_M60_fora,avg_min_M60_casa,avg_min_M60_fora,dp_min_M60_casa,dp_min_M60_fora,cv_min_M60_casa,cv_min_M60_fora,n_marcou_M60_casa,n_marcou_M60_fora,%Gl_Ca_61_70M,%Gl_Fo_61_70M,%Gl_Ca_71_80M,%Gl_Fo_71_80M,%Gl_Ca_81_90M,%Gl_Fo_81_90M
0,Alemanha_1 Bundesliga,Augsburg,14,13,50.0,23.1,77.7,86.7,10.0,11.5,12.9,13.3,7,3,7.1,0.0,28.6,7.7,14.3,30.8
1,Alemanha_1 Bundesliga,Bayer Leverkusen,13,14,53.8,42.9,78.4,82.2,12.1,12.5,15.4,15.2,7,6,23.1,7.1,15.4,14.3,23.1,21.4
2,Alemanha_1 Bundesliga,Bayern München,14,13,85.7,84.6,72.6,74.3,10.2,9.4,14.1,12.7,12,11,57.1,53.8,42.9,30.8,50.0,92.3
3,Alemanha_1 Bundesliga,Borussia Dortmund,14,13,71.4,61.5,84.1,76.8,10.1,12.0,12.1,15.7,10,8,14.3,30.8,21.4,7.7,64.3,38.5
4,Alemanha_1 Bundesliga,Borussia M'gladbach,13,14,38.5,50.0,70.0,78.1,14.2,11.2,20.3,14.3,5,7,30.8,14.3,15.4,21.4,23.1,21.4


In [11]:
# Times que mais marcam apos min 60 jogando em casa
df_M60.nlargest(15, "pct_marcou_M60_casa")[
    ["Liga", "time", "pct_marcou_M60_casa",
     "avg_min_M60_casa", "dp_min_M60_casa", "cv_min_M60_casa",
     "%Gl_Ca_61_70M", "%Gl_Ca_71_80M", "%Gl_Ca_81_90M"]
]

,Liga,time,pct_marcou_M60_casa,avg_min_M60_casa,dp_min_M60_casa,cv_min_M60_casa,%Gl_Ca_61_70M,%Gl_Ca_71_80M,%Gl_Ca_81_90M
157,Chile_1 Primera Division,Huachipato,100.0,78.3,15.2,19.4,33.3,0.0,100.0
162,Chile_1 Primera Division,Universidad Católica,100.0,71.3,1.5,2.1,33.3,66.7,0.0
2,Alemanha_1 Bundesliga,Bayern München,85.7,72.6,10.2,14.1,57.1,42.9,50.0
510,Turquia_1 Super Lig,Galatasaray,85.7,75.2,11.3,15.0,50.0,7.1,71.4
169,Colombia_1 Liga BetPlay,Atlético Nacional,83.3,72.2,6.6,9.2,16.7,66.7,16.7
176,Colombia_1 Liga BetPlay,Fortaleza CEIF,83.3,87.0,7.4,8.5,0.0,16.7,66.7
486,Portugal_1 Primeira Liga,Sporting CP,83.3,76.0,11.3,14.9,25.0,50.0,50.0
52,Alemanha_3 3. Liga,Verl,80.0,73.2,11.2,15.3,46.7,40.0,26.7
80,Argentina_1 Liga Profesional,San Lorenzo,80.0,80.5,10.3,12.8,20.0,0.0,80.0
112,Belgica_1 Pro League,Union Saint-Gilloise,80.0,75.6,9.9,13.1,20.0,46.7,33.3


In [12]:
# Times que mais marcam apos min 60 jogando fora
df_M60.nlargest(15, "pct_marcou_M60_fora")[
    ["Liga", "time", "pct_marcou_M60_fora",
     "avg_min_M60_fora", "dp_min_M60_fora", "cv_min_M60_fora",
     "%Gl_Fo_61_70M", "%Gl_Fo_71_80M", "%Gl_Fo_81_90M"]
]

,Liga,time,pct_marcou_M60_fora,avg_min_M60_fora,dp_min_M60_fora,cv_min_M60_fora,%Gl_Fo_61_70M,%Gl_Fo_71_80M,%Gl_Fo_81_90M
165,Chile_1 Primera Division,Ñublense,100.0,84.3,9.1,10.8,0.0,33.3,66.7
2,Alemanha_1 Bundesliga,Bayern München,84.6,74.3,9.4,12.7,53.8,30.8,92.3
27,Alemanha_2 2. Bundesliga,Hertha BSC,84.6,78.6,10.5,13.4,23.1,38.5,38.5
347,Holanda_2 Eerste Divisie,VVV,82.4,79.1,10.0,12.7,17.6,29.4,41.2
373,Inglaterra_1 Premier League,Liverpool,81.2,83.6,13.1,15.7,18.8,12.5,50.0
43,Alemanha_3 3. Liga,Ingolstadt,80.0,75.0,11.4,15.2,40.0,20.0,46.7
14,Alemanha_1 Bundesliga,Stuttgart,78.6,78.7,8.3,10.5,14.3,42.9,50.0
62,Argentina_1 Liga Profesional,Boca Juniors,75.0,78.3,13.2,16.9,25.0,0.0,50.0
72,Argentina_1 Liga Profesional,Independiente Rivadavia,75.0,76.0,15.7,20.7,50.0,0.0,25.0
162,Chile_1 Primera Division,Universidad Católica,75.0,90.7,6.0,6.6,0.0,0.0,75.0


In [13]:
# Filtrar por liga especifica
liga_filtro = "Inglaterra_1 Premier League"
cols_faixa  = [c for c in df_M60.columns if c.startswith("%Gl_Ca_")]

df_M60[df_M60["Liga"] == liga_filtro][[
    "time", "pct_marcou_M60_casa",
    "avg_min_M60_casa", "dp_min_M60_casa", "cv_min_M60_casa"
] + cols_faixa].sort_values("pct_marcou_M60_casa", ascending=False)

,time,pct_marcou_M60_casa,avg_min_M60_casa,dp_min_M60_casa,cv_min_M60_casa,%Gl_Ca_61_70M,%Gl_Ca_71_80M,%Gl_Ca_81_90M
363,Arsenal,80.0,78.8,11.3,14.3,33.3,20.0,53.3
376,Newcastle United,62.5,80.9,9.7,12.0,12.5,25.0,50.0
378,Sunderland,60.0,76.9,10.8,14.0,13.3,33.3,33.3
362,AFC Bournemouth,56.2,74.6,12.8,17.2,31.2,12.5,37.5
371,Fulham,56.2,78.0,11.9,15.3,12.5,37.5,37.5
375,Manchester United,53.3,76.4,13.5,17.7,20.0,26.7,33.3
365,Brentford,53.3,81.1,12.3,15.1,13.3,20.0,46.7
374,Manchester City,53.3,67.8,9.8,14.5,53.3,0.0,20.0
364,Aston Villa,50.0,77.1,11.5,14.9,18.8,12.5,25.0
366,Brighton & Hove Albion,50.0,78.0,10.6,13.6,18.8,12.5,37.5
